<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/21_frozen_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 21 — Dondurulmuş gömmeler ve lineer prob (EXP-016 / 017 / 018)

**Bilimsel amaç** (D-034, D-035; `docs/PHASE2_DESIGN.md`):
- **EXP-016 (onaylayıcı):** 6 backbone'un füzyon AUC'si, aynı katılımcı düzeyi dış fold'larda ve aynı LR protokolüyle, iki MFCC tabanından (mfcc_lr, mfcc_mlp) yüksek mi? Tek yönlü Nadeau–Bengio, kesişim-birleşim, Holm (6); destek: bootstrap, DeLong. Ayrıca D-035: Faz 3 için fold başına seçim (yalnız o fold'un eğitim verisindeki iç doğrulama).
- **EXP-017 (keşifsel):** BEATs / WavLM katman katman (yalnız tekrar 0).
- **EXP-018 (keşifsel):** tüm kayıt tek girdi vs 4 s pencereler.

**Üst sınır uyarısı (D-028):** Kayıt bağlamı tek başına AUC ≈ 0.93. Bağlam değerlendirmesi model geliştirme bitince yapılacak; bu notebook yalnız izleme göstergelerini raporlar.

**Bu notebook Faz 3'ün yerine geçmez.** Faz 2, ham ses üzerinde uçtan uca fine-tune'dan (Faz 3, ana hedef) önce kontrollü bir temsil karşılaştırmasıdır.

**Ön koşul:** 20 numaralı notebook'un açtığı SMK-001 PR'ı **main'e birleştirilmiş** ve her backbone **PASS**. Aşağıdaki kapı hücresi bunu, SMK-001'den sonra backbone kodunun değişmediğini ve kullanılan checkpoint'lerin SMK-001'dekilerle aynı olduğunu kontrol eder.

**Çalışma ortamı:**
- Bölüm A (gömme çıkarımı): *T4 GPU*, toplam < 1 saat [tahmin; SMK-001 S13'e bak]. Gömmeler doğrudan Drive'a yazılır; kopma olursa hücreyi yeniden çalıştır, biten parçalar atlanır.
- Bölüm B (problar): CPU yeterli (GPU oturumunu boşa harcamamak için ayrı bir CPU oturumunda çalıştırabilirsin: 0. bölümü çalıştır, A'yı atla, B'den devam et). EXP-016 birkaç saat, EXP-017 ~1–2 saat [tahmin]. Ara sonuçlar Drive `experiments/frozen_probe/partial/` altında; yeniden çalıştırınca biten (kol, tekrar) çiftleri atlanır.

## 0. Oturum kurulumu

In [ ]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). Bölüm A (gömme çıkarımı) GPU ister; Bölüm B için CPU yeterli.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
BACKBONES   = ['cnn10', 'cnn14', 'cnn14_16k', 'beats', 'wavlm_base_plus', 'wavlm_large']
EXPECTED_N_RECORDINGS, EXPECTED_N_PARTICIPANTS = 2393, 342   # Faz 1 önbellek kohortu (reports/audio_cache)
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

In [ ]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

In [ ]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
import torch, torchaudio, transformers, torchlibrosa, sklearn, numpy as np
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__,
      '| sklearn', sklearn.__version__, '| numpy', np.__version__)
import torchaudio.compliance.kaldi  # BEATs bunu gerektirir; yoksa burada hata verir (sözleşme notu)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Kapı: SMK-001 raporu main'de, tamamı PASS, ve SMK-001'den sonra backbone kodu değişmemiş
SMK_PATH = 'reports/phase2/SMK-001_smoke.json'
if git('ls-files', '--error-unmatch', SMK_PATH, check=False).returncode != 0:
    raise RuntimeError("SMK-001 raporu main dalında yok. 20 numaralı notebook'un açtığı phase2-smoke-results-… PR'ını "
                       "GitHub'da main'e birleştir, sonra bu notebook'ta 0. bölümü yeniden çalıştır.")
assert not git('status', '--porcelain', '--', SMK_PATH).stdout.strip(), 'SMK-001 raporu yerelde değiştirilmiş'
smk = json.loads((REPO_DIR / SMK_PATH).read_text())
status = {r['backbone']: r['status'] for r in smk['results']}
print('SMK-001:', smk['date'], status, '| random_init:', smk['random_init'])
assert smk['exp'] == 'SMK-001' and not smk['random_init']
assert set(BACKBONES) <= set(status), f'SMK-001 raporunda eksik backbone: {set(BACKBONES) - set(status)}'
assert all(status[b] == 'PASS' for b in BACKBONES), 'SMK-001 tamamen PASS değil → çıkarım yapma (docs/PHASE2_DESIGN.md 1.1)'
assert smk.get('code_sha256'), ('SMK-001 raporunda code_sha256 yok (eski sürümle üretilmiş): SMK-001\'i güncel 20 numaralı '
                                'notebook ile yeniden çalıştır.')
changed = [p for p, h in smk['code_sha256'].items() if sha256(REPO_DIR / p) != h]
assert not changed, f'SMK-001\'den sonra değişen backbone kodu: {changed} → SMK-001\'i yeniden çalıştır'
SMK_CKPT = {r['backbone']: r['checks']['S1_sha256']['sha256'] for r in smk['results'] if r['backbone'] in BACKBONES}
print('Kapı geçti: 6 backbone PASS; backbone kodu SMK-001 ile aynı (commit', smk.get('git_commit', '')[:12] + ').')

## A. Gömme çıkarımı (GPU)
Her backbone için `win4` (birincil) ve `full` (EXP-018) görünümleri, fp32, bütün katmanlar. Çıktı: Drive `data_derived/embeddings_v1/<backbone>/` (katılımcı düzeyi → git'e girmez). `MANIFEST.sha256` varsa backbone atlanır.

**Kısa kayıt politikası (D-034 madde 1 — KABUL: zeropad):** 4 s'den kısa 17 kayıt için iki politika da hesaplanır; `SHORT_POLICY` birincil olanı seçer, diğeri `short_alt_win4.npz`'ye yazılır ve EXP-016S'de duyarlılık olarak kullanılır. `zeropad` = D-015 (Boll ile aynı; Faz 3 giriş politikasıyla tutarlı). Bunun dolgu kaynaklı süre ipuçlarını ortadan kaldırdığı varsayılmaz (META-016, EXP-016S). **Gerçek sonuçlara bakarak değiştirme.**

In [ ]:
SHORT_POLICY = 'zeropad'   # D-034 KABUL (13. tur): birincil zeropad; nopad EXP-016S'de duyarlılık. Değiştirme.
assert SHORT_POLICY in {'zeropad', 'nopad'}
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'Bölüm A GPU ister: Runtime → Change runtime type → T4 GPU'

In [ ]:
# Drive → yerel disk (okuma hızı). Yarım kalmış kopya .part olarak kalır ve bir sonraki çalıştırmada yeniden kopyalanır.
info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
LOCAL_CACHE.mkdir(exist_ok=True)
for name in ['audio_32k.f32', 'audio_16k.f32', 'index.csv']:
    dst = LOCAL_CACHE / name
    if dst.exists() and sha256(dst) != info['files_sha256'][name]:
        print(name, ': yerel kopya bozuk, yeniden kopyalanıyor'); dst.unlink()
    if not dst.exists():
        tmp = dst.with_name(name + '.part')
        shutil.copy(DRIVE_CACHE / name, tmp)
        tmp.rename(dst)
    assert sha256(dst) == info['files_sha256'][name], f'{name}: sha256 cache_info.json ile uyuşmuyor (Drive kopyası bozuk?)'
print('Önbellek yerel diskte ve doğrulandı.')

In [ ]:
for b in BACKBONES:
    out = EMB_DIR / b
    if (out / 'MANIFEST.sha256').exists():
        print(b, ': mevcut, atlandı'); continue
    sh(f'cd {REPO_DIR} && python scripts/extract_embeddings.py --backbone {b} --model-dir {MODELS} --cache-dir {LOCAL_CACHE} '
       f'--out-dir {out} --views win4,full --short-policy {SHORT_POLICY} --device {DEVICE}')
    assert (out / 'MANIFEST.sha256').exists(), f'{b}: çıkarım tamamlanmadı'

In [ ]:
import pandas as pd
rows = []
for b in BACKBONES:
    i = json.loads((EMB_DIR / b / 'info.json').read_text())
    rows.append({'backbone': b, 'n_recordings': i['n_recordings'], 'n_participants': i['n_participants'], 'n_windows': i['n_windows_total'],
                 'n_short': i['n_short_lt4s'], 'shape': i['shapes']['rec_win4.npy'], 'n_params_M': round(i['n_params'] / 1e6, 1),
                 'ckpt_sha256': i.get('checkpoint_sha256', '')[:12], 'min': round(i['elapsed_s'] / 60, 1)})
    assert i['n_recordings'] == EXPECTED_N_RECORDINGS and i['n_participants'] == EXPECTED_N_PARTICIPANTS and not i['random_init']
    assert i['short_policy'] == SHORT_POLICY, f"{b}: gömmeler başka politikayla çıkarılmış ({i['short_policy']})"
    assert i['checkpoint_sha256'] == SMK_CKPT[b], f'{b}: gömmeler SMK-001\'de doğrulanan checkpoint ile çıkarılmamış'
pd.DataFrame(rows)

## B. Problar (CPU yeterli)
Ayrı bir CPU oturumunda başlıyorsan: önce 0. bölümü çalıştır, sonra buradan devam et (A gerekmez).

Sıra önemli: EXP-016 önce (sonrakiler EXP-016'nın ortak ara sonuçlarını yeniden kullanır).
- **EXP-016:** önceden belirlenmiş onaylayıcı analiz + D-035 fold başına seçim (`experiments/frozen_probe/d035_selection.csv`).
- **EXP-016S:** önceden belirlenmiş duyarlılık (kısa kayıtlarda diğer dolgu politikası; tekrar 0).
- **META-016:** ayrı meta veri / confounder izleme raporu (bağlam bağımlılığı, süre); ana akışa girmez.
- **EXP-017 / EXP-018:** keşifsel; hiçbir seçimde kullanılmaz.

In [ ]:
MFCC = DERIVED / 'mfcc' / 'features_cache.csv'
assert MFCC.exists(), 'EXP-011 özellik dosyası (features_cache.csv) yok'
for b in BACKBONES:
    assert (EMB_DIR / b / 'MANIFEST.sha256').exists(), f'{b}: gömmeler yok → önce Bölüm A (GPU)'
# META-016 önbellekten yalnız index.csv'yi okur: Drive kopyası doğrulanıp doğrudan kullanılır (CPU oturumunda A gerekmez)
assert sha256(DRIVE_CACHE / 'index.csv') == json.loads((DRIVE_CACHE / 'cache_info.json').read_text())['files_sha256']['index.csv']
COMMON = (f'--emb-root {EMB_DIR} --context {DERIVED}/participant_context.csv --splits-dir {DERIVED}/splits '
          f'--out-dir {EXPERIMENTS}/frozen_probe --report-dir {REPO_DIR}/reports/frozen --registry {REPO_DIR}/results/registry.csv')

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-016 --mfcc-features {MFCC} {COMMON}')

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-016S {COMMON}')

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/report_metadata_monitor.py --probe-dir {EXPERIMENTS}/frozen_probe --cache-dir {DRIVE_CACHE} '
   f'--context {DERIVED}/participant_context.csv --splits-dir {DERIVED}/splits --report-dir {REPO_DIR}/reports/frozen')

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-017 {COMMON}')

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-018 {COMMON}')

## C. Raporları GitHub'a gönder (yeni dal)
Yalnız `reports/frozen/` ve `results/registry.csv` (gönderimden önce katılımcı ID'si kontrol edilir). Gömmeler ve katılımcı düzeyi tahminler Drive'da kalır (D-014). Dal adı zaman damgalıdır; gönderimden sonra repo `main`'e döner. Notebook'u kaydetmeden önce çıktıları temizle.

In [ ]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'phase2-frozen-results', ['reports/frozen', 'results/registry.csv'], 'EXP-016/017/018: dondurulmuş gömme + lineer prob (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')